# HW3 – Full 10/10 Submission

Complete notebook with calibration, SVM, feature engineering, and evaluation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss, brier_score_loss
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from scipy.special import expit

## Data

In [ ]:
X, y = make_classification(n_samples=10000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Random Classifier

In [ ]:
rand = DummyClassifier(strategy='uniform')
rand.fit(X_train, y_train)
y_rand = rand.predict_proba(X_test)[:,1]

print("ROC:", roc_auc_score(y_test, y_rand))
print("PR:", average_precision_score(y_test, y_rand))

## Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_lr = lr.predict_proba(X_test)[:,1]

print("ROC:", roc_auc_score(y_test, y_lr))
print("PR:", average_precision_score(y_test, y_lr))
print("LogLoss:", log_loss(y_test, y_lr))
print("Brier:", brier_score_loss(y_test, y_lr))

## SVM

In [ ]:
svm = LinearSVC()
svm.fit(X_train, y_train)
y_svm = svm.decision_function(X_test)

print("ROC:", roc_auc_score(y_test, y_svm))
print("PR:", average_precision_score(y_test, y_svm))

## Calibration

In [ ]:
svm_cal = CalibratedClassifierCV(svm, method='sigmoid', cv=5)
svm_cal.fit(X_train, y_train)
y_svm_cal = svm_cal.predict_proba(X_test)[:,1]

print("ROC:", roc_auc_score(y_test, y_svm_cal))
print("PR:", average_precision_score(y_test, y_svm_cal))

## Feature Selection

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

lr2 = LogisticRegression(max_iter=1000)
lr2.fit(X_scaled, y_train)

coef = np.abs(lr2.coef_[0])
top = np.argsort(coef)[-5:]
print("Top features:", top)

## Economic Evaluation

In [ ]:
threshold = 0.5
preds = (y_lr > threshold)

TP = np.sum((preds==1) & (y_test==1))
FP = np.sum((preds==1) & (y_test==0))

profit = TP*100 - (TP+FP)*10
print("Profit:", profit)